In [1]:
!pip install recbole

In [2]:
!pip install kmeans-pytorch

In [3]:
!pip install ray


In [4]:
import pandas as pd
from recbole.quick_start import run_recbole
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import BPR
from recbole.model.sequential_recommender import SASRec
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger



RecBole ожидает, что датасет Yambda будет лежать по пути data_path/dataset_name/dataset_name.inter (формат Atomic Files)

В recbole :

взаимодействия  .inter

характеристики (embeddings.parquet) - .item

файл multi_event.parquet объединяет лайки, дизлайки и прослушивания в один поток

In [5]:
import os
import pandas as pd
from datasets import load_dataset

# папка для датасета
os.makedirs('yamda', exist_ok=True)

ds = load_dataset("yandex/yambda", data_dir="flat/50m", data_files= "multi_event.parquet", split='train')

df = ds.to_pandas()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
df.columns


Index(['uid', 'timestamp', 'item_id', 'is_organic', 'played_ratio_pct',
       'track_length_seconds', 'event_type'],
      dtype='object')

Теперь сделаем три выборки для эксперимента



In [7]:
import os

datasets = ['yambda_organic', 'yambda_rec', 'yambda_random']
for d in datasets:
  os.makedirs(d,exist_ok=True)

In [8]:
os.makedirs('yamda_organic', exist_ok=True)
os.makedirs('yamda_rec', exist_ok=True)
os.makedirs('yamda_random', exist_ok=True)

In [9]:
def save_inter_file(data, dataset_name):

    valid_data = data[
        (data['event_type'] == 'like') |
        (data['played_ratio_pct'] > 50)
    ].copy()

    inter_df = valid_data[['uid', 'item_id', 'timestamp']]
    inter_df.columns = ['user_id:token', 'item_id:token', 'timestamp:float']

    file_path = f"{dataset_name}/{dataset_name}.inter"
    inter_df.to_csv(file_path, sep='\t', index=False)
    print(f"Файл {file_path} создан. Строк: {len(inter_df)}")

save_inter_file(df[df['is_organic'] == True], 'yamda_organic')
save_inter_file(df[df['is_organic'] == False], 'yamda_rec')
save_inter_file(df, 'yamda_random')

Файл yamda_organic/yamda_organic.inter создан. Строк: 14584508
Файл yamda_rec/yamda_rec.inter создан. Строк: 15667339
Файл yamda_random/yamda_random.inter создан. Строк: 30251847


In [10]:
target_size = 14584508
seed = 42

datasets = ['yamda_organic', 'yamda_rec', 'yamda_random']

for d in datasets:
    file_path = f"{d}/{d}.inter"

    temp_df = pd.read_csv(file_path, sep='\t')

    if len(temp_df) > target_size:
        print(f"Обрезка {d}: {len(temp_df)} -> {target_size}")
        temp_df = temp_df.sample(n=target_size, random_state=seed)

        temp_df.to_csv(file_path, sep='\t', index=False)
        print(f"Файл {file_path} успешно обновлен.")
    else:
        print(f"Файл {d} уже имеет минимальный размер или меньше.")

Файл yamda_organic уже имеет минимальный размер или меньше.
Обрезка yamda_rec: 15667339 -> 14584508
Файл yamda_rec/yamda_rec.inter успешно обновлен.
Обрезка yamda_random: 30251847 -> 14584508
Файл yamda_random/yamda_random.inter успешно обновлен.


## 1 экперимент

Для Global Timeline Splitting

укажем

group_by: None

In [11]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.sequential_recommender import SASRec
from recbole.model.general_recommender import BPR
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

In [12]:
dataset_name = 'yamda_organic'

parameter_dict = {
    'data_path': './',
    'dataset': dataset_name,
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'TIME_FIELD': 'timestamp',
    'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},

    #  ГЛОБАЛЬНОЕ ВРЕМЕННОЕ РАЗДЕЛЕНИЕ
    'eval_args': {
        'split': {'RS': [0.8, 0.1, 0.1]}, # 80% обучение, 10% валидация, 10% тест
        'group_by': None,                 # None делает сплит ГЛОБАЛЬНЫМ по времени
        'order': 'TO',                    # TO = Time Order (сортировка по времени перед разделением)
        'mode': 'full'                    # Ранжирование по всем доступным объектам
    },

    # параметры обучения
    'train_batch_size': 2048,
    'epochs': 10,
    'learning_rate': 0.001,
    'metrics': ['Recall', 'NDCG'],
    'topk': [10, 20],
    'device': 'CUDA',
    'reproducibility': True,
    'seed': 42
}


In [13]:
!pip install "numpy<2.0"


In [14]:
# Разделение для подвыборки только с органическими взаимодействиями ---
config_init = Config(model='SASRec', config_dict=parameter_dict)
init_seed(config_init['seed'], config_init['reproducibility']) #перемешивать даннеы  всегда одинаково
dataset = create_dataset(config_init)
train_data, valid_data, test_data = data_preparation(config_init, dataset)


ValueError: train_neg_sample_args [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}] should be None when the loss_type is CE.

In [ ]:
# Обучение BPR
config_bpr = Config(model_name='BPR', config_dict=parameter_dict)
model_bpr = BPR(config_bpr, train_data.dataset).to(config_bpr['device'])
trainer_bpr = Trainer(config_bpr, model_bpr)
best_valid_score_bpr, best_valid_result_bpr = trainer_bpr.fit(train_data, valid_data)
test_result_bpr = trainer_bpr.evaluate(test_data)

# Обучение SASRec
print("=== Подготовка SASRec ===")
config_sas = Config(model_name='SASRec', config_dict=parameter_dict)
model_sas = SASRec(config_sas, train_data.dataset).to(config_sas['device'])
trainer_sas = Trainer(config_sas, model_sas)
best_valid_score_sas, best_valid_result_sas = trainer_sas.fit(train_data, valid_data)
test_result_sas = trainer_sas.evaluate(test_data)

# Сравнение
print("BPR:", test_result_bpr)
print("SASRec:", test_result_sas)


In [15]:
# Упаковываем папки в архив data.zip
!zip -r data.zip yamda_organic yamda_rec yamda_random


  adding: yamda_organic/ (stored 0%)
  adding: yamda_organic/yamda_organic.inter (deflated 75%)
  adding: yamda_rec/ (stored 0%)
  adding: yamda_rec/yamda_rec.inter (deflated 55%)
  adding: yamda_random/ (stored 0%)
  adding: yamda_random/yamda_random.inter (deflated 55%)
